# ezTrack: Location Tracking (single video)

Track one animal across a single video: load it, crop, exclude regions, build a background reference, (optionally) define regions of interest and a real-world scale, then track and summarize.

Every selection you make below (crop, mask, ROIs, scale) is stored on `session.selections`, a plain serializable object. You can `session.selections.save(...)` and reload it later to replay the exact same analysis (see step 8).

## 1. Load packages

In [ ]:
import holoviews as hv
import eztrack as ez

hv.extension("bokeh")

## 2. Point at the video
Run the next cell to get a file picker, then browse to and select your video. (It browses the Jupyter kernel's filesystem, so it works the same on a local or remote server.) `start`/`end` are frame numbers (`end=None` runs to the end); set them in step 2b. You'll draw and **name** any regions of interest later, in step 6.

Prefer to type the path instead? Skip the picker and set `dpath`/`file` directly in step 2b. On Windows, prefix raw paths with `r`, e.g. `r"C:\Videos"`.

In [ ]:
fc = ez.file_chooser("")  # starts in the working directory; browse to your video
fc

### 2b. Build the session
Reads the file you picked above. To type a path by hand instead, replace `fc.selected_path` / `fc.selected_filename` with `dpath="..."` / `file="..."`.

In [ ]:
session = ez.Session(
    dpath=fc.selected_path,
    file=fc.selected_filename,
    start=0,
    end=None,
    # Speed knobs, as reduction factors (1 = off). They only trade a little
    # precision/coverage for a lot of speed; outputs stay in the original space:
    #   spatial_downsample=2  -> track at half resolution (positions still
    #                            reported in full-res pixels)
    #   temporal_downsample=2 -> track every other frame; the output then has a
    #                            row per *tracked* frame (the `frame` column keeps
    #                            the real frame numbers, no positions are invented)
    spatial_downsample=1,
    temporal_downsample=1,
    # Regions of interest are drawn and named in step 6, no need to declare them
    # here. (You *can* preset region_names=[...] to fix the count/labels up front.)
    region_names=None,
)

# Where outputs go: a sibling `eztrack_out/` next to the input video folder,
# so they land at <session>/eztrack_out/, the layout the workshop capstone
# reads. Set OUT_DIR elsewhere to taste (e.g. Path(session.dpath) to save the
# track right beside the video).
from pathlib import Path
OUT_DIR = Path(session.dpath).parent / "eztrack_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load video and crop (optional)
Run the cell, then use the box-select tool beneath the image to draw the region to **keep**. Re-run this cell (and the steps after it) if you change the crop.

In [ ]:
hv.output(size=50)

ez.crop_tool(session)

## 4. Exclude internal regions (optional)
Draw any regions to **exclude** from analysis (e.g. a cable port). Double-click to start a polygon, single-click to add vertices, double-click to close.

In [ ]:
hv.output(size=100)

ez.mask_tool(session)

## 5. Build the reference (background) frame
The reference is the per-pixel **median** of frames sampled across the session, which removes the moving animal. Works well when the animal isn't still for >50% of the session.

**Alternative:** if you have an animal-free clip of the same arena, set `session.altfile = "empty.avi"` before calling `ez.reference_frame(...)` and it will sample that clip instead of the analyzed video.

In [ ]:
hv.output(size=100)

ez.reference_frame(session, num_frames=50)
ez.image(session.reference, session.stretch, "Reference Frame", colorbar=True)

## 6. Define regions of interest (optional)
Draw each region with the polygon tool (double-click to start, single-click to add vertices, double-click to close). Overlapping regions are fine. They're labeled `zone_1`, `zone_2`, and so on as you draw; you'll give them real names in **step 6b**. Skip both cells if you don't need ROIs.

In [ ]:
hv.output(size=100)

ez.roi_tool(session)

### 6b. Name the regions
Give one name per region you drew, **in the order you drew them**. The count must match (you'll get an error if it doesn't), and the cell redraws the arena with your labels so you can confirm the order.

In [ ]:
hv.output(size=100)

ez.name_rois(session, ["left", "right", "top", "bottom"])

## 7. Define a real-world scale (optional)
Click two points a known distance apart, then record that distance and its unit. Tracking output will include a `distance_<unit>` column.

In [ ]:
hv.output(size=100)

ez.distance_tool(session)

In [ ]:
ez.set_scale(session, real_distance=100, unit="cm")

## 8. Save the selections (optional)
Persist crop + mask + ROIs + scale to a JSON file so this analysis can be replayed exactly (and reused across videos).

In [ ]:
session.selections.save(str(OUT_DIR / "selections.json"))
# later, or in another notebook:
# session.selections = ez.Selections.load(str(OUT_DIR / "selections.json"))

## 9. Tune tracking parameters
Every `TrackParams` option is spelled out in the next cell. What each one does:

- **`threshold_pct`**: keep only the brightest X% of frame-vs-reference differences (higher = stricter; fewer pixels survive). This is the default thresholding mode (used when `threshold_abs` is `None`).
- **`threshold_abs`**: an alternative to the percentile, a **fixed cutoff** in 0-255 intensity units (overrides `threshold_pct` when set; leave `None` for the percentile). A fixed cutoff is steadier across frames than a percentile when the amount of visible animal varies (e.g. when a cable partly occludes it).
- **`threshold_on`**: what `threshold_abs` is measured against (only relevant when `threshold_abs` is set; the percentile is always taken on the baseline difference):
    - `"difference"` *(default)*: the change **from the baseline** (reference frame, step 5). Static scene structure (other dark objects, shapes, the cable port, dark corners) is in the baseline and so stays invisible; only changes count. `method` picks the direction: `dark` keeps pixels that dropped at least the cutoff below baseline, `light` those that rose above it, `abs` either way. Best for cluttered arenas.
    - `"raw"`: the **raw pixel value**, ignoring the baseline. `dark` keeps pixels darker than the cutoff value, `light` brighter (no `abs`, since there's no baseline to deviate from). Uniform across the arena, so it doesn't sag in regions where the baseline is already near the animal's brightness, but it flags *every* static object at that brightness, so mask those out in step 4. Good when the animal is reliably the darkest/brightest thing in the (masked) arena.
- **`method`**: `"abs"` (any change), `"light"` (animal lighter than the background), or `"dark"` (animal darker than the background).
- **`window`**: bias the estimate toward the previous location so distractors elsewhere are ignored. `Window(size=..., weight=...)`: `size` is the search box in px, `weight` (0-1) how strongly to favor it. Set `window=None` to disable.
- **`denoise`** / **`denoise_kernel`**: morphologically **open** the thresholded mask each frame (erode then dilate). This drops specks and thin features (e.g. a headstage wire) smaller than `denoise_kernel` px while keeping the animal's bulk. Off by default; turn on with `denoise=True` and raise `denoise_kernel` to remove larger bits of noise.

The preview shows tracking on a few **random** frames, **shown as-is (successes *and* failures)**: each original beside its thresholded difference (after denoising, if enabled), with the detected center of mass circled. It also prints a quick **detection rate**, the % of a larger random sample (`sample=200` by default) where the animal was found, so you can judge how often the params actually work instead of eyeballing a few lucky frames. Tweak the values, re-run, and aim for both a high detection rate **and** markers that land on the animal.

- Want just the number while iterating fast? Call `ez.detection_scan(session, params)` (or pass `sample=0` to the preview to skip the scan).
- Once params roughly work and you only want to inspect good frames, pass `prefer_detected=True`; the printed rate stays honest either way.

(Position **outlier removal** is a separate post-tracking step; see step 10b, after the track exists.)

In [ ]:
params = ez.TrackParams(
    threshold_pct=99.5,  # used only when threshold_abs is None
    threshold_abs=None,  # fixed cutoff (0-255); overrides the percentile when set
    threshold_on="difference",  # what threshold_abs measures: "difference" (vs baseline) | "raw" (pixel value)
    method="abs",  # "abs" | "light" | "dark"  (raw mode needs "dark" or "light")
    window=ez.Window(size=100, weight=0.9),  # or window=None to disable
    denoise=False,  # True = remove small specks / thin wire from the mask
    denoise_kernel=5,  # opening kernel in px (only used when denoise=True)
)

hv.output(size=100)
# Random frames shown as-is (failures included) + a printed detection rate over
# `sample` frames. prefer_detected=True biases only the shown panels, not the rate.
ez.threshold_preview(session, params, examples=4, sample=200, prefer_detected=False)

## 10. Track location
Runs over the whole video. **Too slow?** Bump the speed knobs in the session (step 2b) and re-run from there: `spatial_downsample=2` (~2× faster) and/or `temporal_downsample=4` (~4× faster); they stack (2 + 4 ≈ 10× faster). Positions stay in original pixels. Note `temporal_downsample` gives one output row per *tracked* frame (so the table is shorter, but the `frame` column tells you exactly which frames); there's no fabricated/interpolated data.

The cell also prints the **detection rate** (the % of frames where the animal was actually found). Treat it as a quality check: a low value means much of the track was carried-forward rather than measured. Timing stays out of ezTrack on purpose; the `frame` column is your join key against a per-frame timestamp file (e.g. `timestamps.csv`) downstream.

In [ ]:
location = ez.track(session, params)
location_filtered = location  # no outlier filter yet; step 10b can replace this

# Detection rate: the % of frames the animal was actually found. A low % means much
# of the track is carried-forward (frozen) rather than measured, and distance
# (counted only between detected frames) will under-report; re-tune step 9 if so.
# track() also warns automatically when detection drops below 90%.
print(ez.detection_rate(location))
location.head()

### 10b. Remove position outliers (optional)
Now that the track exists, clean up outlier jumps in the path. The **Hampel filter** looks at each frame's `(x, y)` together and, if it sits more than `sigma` robust-deviations from the median **position** of the `±window` rows around it, replaces *both* coordinates with that local median, killing the odd bad-frame jump without smoothing genuine movement.

`window` is counted in **tracked rows**, not original frames: with `temporal_downsample=N`, each row is N frames apart, so `window=7` spans 15·N frames of real time (spatial downsampling doesn't affect it). Drop `window` if you've temporally downsampled and the filter feels too aggressive.

This filters from the raw `location` every time, so you can freely re-tune `window`/`sigma`; `location` itself is never overwritten. Two views are shown: a **time plot** of `x`, `y`, and the per-frame correction distance (how far each point was moved), and the two **paths side by side** (colored by time). The steps below use `location_filtered` (which equals `location` if you skip this).

In [ ]:
location_filtered = ez.hampel_filter(location, session, window=7, sigma=3.0)

# Time view: x, y, and the correction distance (how far each frame was moved) vs frame.
hv.output(size=100)
ez.outlier_plot(location, location_filtered)

In [ ]:
# Spatial view: the two paths side by side (colored by time).
hv.output(size=80)
(
    ez.trace(session, location, width=340).opts(title="raw")
    + ez.trace(session, location_filtered, width=340).opts(title="filtered")
)

### 10c. Save the track
Write the filtered track to `position.csv` in `OUT_DIR` (the `eztrack_out/` folder next to your video, from step 2b), with a `position.json` sidecar of the run parameters and ROI geometry so the track stays reproducible.

In [ ]:
# Saves position.csv + position.json (the run parameters and ROI
# geometry from df.attrs) so the analysis stays reproducible; use ez.save_tracking,
# not df.to_csv, which would drop that metadata. Timing is left to downstream: join
# the CSV on its `frame` column against your per-frame timestamp file (e.g.
# timestamps.csv); ezTrack never infers time from the video's fps.
ez.save_tracking(location_filtered, str(OUT_DIR / "position.csv"))

## 11. Visualize
Two views: a **motion trace** (the path drawn as a connected line **colored by time**, dark = start → bright = end, overlaid on the reference with ROI outlines) and a smoothed occupancy heatmap. Both fit the notebook width; pass `width=...` to either to resize.

In [ ]:
hv.output(size=100)

ez.trace(session, location_filtered) + ez.heatmap(session, location_filtered)

## 12. Summarize
Total distance traveled and the proportion of time spent in each ROI. Pass `bins` to split the session into named frame ranges, e.g. `{"first_half": (0, 4500), "second_half": (4501, 9000)}`.

In [ ]:
summary = ez.summarize(location_filtered, session, bins=None)
summary

## 13. Play back the tracking (optional)
Replays a segment inline with the estimated location marked. Set `save=True` to also write `video_output.avi`.

In [ ]:
ez.play_inline(
    session,
    ez.PlayParams(start=0, stop=200, fps=30, save=False),
    location_filtered,
)